# Automated PLE Scan over predefined XY Points

This notebook iterates over a list of N coordinates in the Confocal (XY) scan range, commands the scanner to move to each point, reads the DL Pro wavelength via the Wavemeter, performs a PLE scan, reads the Wavemeter again, and saves the data using the PLE GUI's built-in save methodology to ensure paths and logs are synced.

In [ ]:
import time

# ── qudi modules available in the Jupyter kernel namespace ────────
confocal     = galvo_scanning_probe_logic
ple_gui_app  = ple_gui
ple_scan     = laser_scanner_logic
ple_data     = ple_data_logic
wavemeter    = ws_wavemeter
dl_pro_laser = dl_pro

print("Modules successfully linked.")

## 1. Setup Coordinates

In [ ]:
# Define your points here in meters. For example, 1 µm = 1e-6
POINTS_XY = [
    (-10e-6,  5.0e-6),   # Point 1
     (-10e-6,  0e-6),   # Point 1
    (-10e-6,  -5e-6),   # Point 2
    (10e-6, 5e-6),    # Point 3
     (10e-6, 0e-6),    # Point 3
    (10e-6,  -5e-6),   # Point 1
   
]

# Number of lines per PLE scan (used to set the GUI parameter)
PLE_LINES_PER_SCAN = 65

# Voltage shift applied to the DLC PRO laser for lower and higher passes
# (Adjust this to match the voltage equivalent of your ~22 GHz range)
LASER_VOLTAGE_SHIFT_V = 20.0

print(f"{len(POINTS_XY)} points defined.")

## 2. Helper Functions

In [19]:
def get_wavelength(wm, retries=10):
    """Attempts to read the current wavelength from the WS wavemeter."""
    val = -1.0
    for attempt in range(retries):
        try:
            if hasattr(wm, 'get_current_wavelength'):
                val = float(wm.get_current_wavelength())
            elif hasattr(wm, 'get_wavelength'):
                # Try specific channel or default
                val = float(wm.get_wavelength())
                
            if val > 0:
                break
        except Exception:
            pass
        time.sleep(0.1)
    return val

## 3. Execution Loop

In [16]:
_stop_flag = False

def run_automation():
    global _stop_flag
    _stop_flag = False
    
    print(f"Starting Automated Scan Sequence...")
    
    # Read the current central voltage from the DLC Pro laser
    orig_laser_voltage = dl_pro_laser.get_pc_voltage_set()
    
    # Three passes defined by their offset multipliers
    # 0 = Central, -1 = Lower voltages (-shift), 1 = Higher voltages (+shift)
    voltage_passes = [
        ('Central',  0),
        ('Lower',   -1),
        ('Higher',   1)
    ]
    
    for pass_name, offset_multi in voltage_passes:
        if _stop_flag:
            print("\n--- Sequence Interrupted by User ---")
            break
            
        new_laser_voltage = orig_laser_voltage + (offset_multi * LASER_VOLTAGE_SHIFT_V)
        
        print(f"\n{'='*50}")
        print(f"=== Starting PASS: {pass_name} ===")
        print(f"=== Laser PC Voltage Set to: {new_laser_voltage:.3f} V ===")
        print(f"{'='*50}\n")
        
        # Apply new voltage range to the hardware
        dl_pro_laser.set_pc_voltage(new_laser_voltage)
        time.sleep(2.0) # give laser time to stabilize at new voltage
        
        for idx, (x, y) in enumerate(POINTS_XY):
            if _stop_flag:
                print("\n--- Sequence Interrupted by User ---")
                break
                
            # Include the pass_name in the tag so records aren't overwritten
            tag = f"P_{idx:03d}_{pass_name}_X{x*1e6:.2f}_Y{y*1e6:.2f}"
            print(f"\n[{pass_name} Pass] [Point {idx+1}/{len(POINTS_XY)}] Moving to X: {x*1e6:.2f} µm, Y: {y*1e6:.2f} µm")
            
            # 1. Move Confocal Scanner
            confocal.set_target_position({'x': x, 'y': y}, move_blocking=True)
            time.sleep(1.0) # wait for settling
            
            # 2. Get Starting Wavelength
            start_wl = get_wavelength(wavemeter)
            print(f"   --> Start Wavelength: {start_wl:.5f} nm")
            
            # 3. Apply PLE lines setting in GUI
            ple_gui_app._mw.number_of_repeats_SpinBox.setValue(PLE_LINES_PER_SCAN)
            ple_gui_app._mw.number_of_repeats_SpinBox.editingFinished.emit()
            time.sleep(0.2)
            
            # 4. Trigger PLE Scan from GUI
            print(f"   --> Running PLE scan...")
            ple_gui_app._mw.actionToggle_scan.setChecked(True)
            ple_gui_app.toggle_scan()
            time.sleep(1.0)
            
            # 5. Wait for PLE scan to complete
            while ple_scan.module_state() != 'idle':
                if _stop_flag:
                    ple_scan.stop_scan()
                    break
                time.sleep(0.5)
                
            ple_gui_app._mw.actionToggle_scan.setChecked(False)
            
            if _stop_flag:
                continue
            
            # 6. Get Stopping Wavelength
            stop_wl = get_wavelength(wavemeter)
            print(f"   --> Stop Wavelength:  {stop_wl:.5f} nm")
            
            # 7. Collect and Save Data using the PLE GUI app method
            scan_data = ple_gui_app.scan_data
            if scan_data is not None:
                # Update the text field in GUI so it visually reflects the tag
                ple_gui_app.save_path_widget.saveTagLineEdit.setText(tag)
                
                # Extract standard parameters from GUI (colors, paths via checkboxes)
                cbar_range = ple_gui_app._mw.matrix_widget.image_widget.levels
                
                if ple_gui_app.save_path_widget.DailyPathCheckBox.isChecked():
                    folder = None
                    ple_gui_app.save_path_widget.currPathLabel.setText("Default")
                else:
                    folder = ple_gui_app._save_folderpath
                    
                # Combine Controller widget metadata (if any) with wavemeter info
                meta = {}
                if ple_gui_app._controller_logic is not None:
                    meta.update(ple_gui_app._mw.Controller_widget.params)
                    
                meta['voltage_pass'] = pass_name
                meta['laser_pc_voltage'] = new_laser_voltage
                meta['automated_point_index'] = idx
                meta['confocal_target_x_m'] = x
                meta['confocal_target_y_m'] = y
                meta['wavemeter_start_wl_nm'] = start_wl
                meta['wavemeter_stop_wl_nm'] = stop_wl

                print(f"   --> Triggering GUI save with tag: {tag}")
                # Emit the GUI's signal to save the data exactly as clicking "Save Data" would!
                ple_gui_app.sigSaveScan.emit(
                    scan_data,
                    ple_gui_app._scanning_logic._channel,
                    ple_gui_app._scanning_logic._fit_container,
                    cbar_range,
                    tag,
                    folder,
                    meta
                )
                print(f"   --> Save triggered to pathway: {'Default' if folder is None else folder}")
            else:
                print(f"   --> [ERROR] No scan data found in ple_gui_app to save for point {idx+1}.")
                
            time.sleep(0.5) # pause between points
            
    print("\nDone!\n")
    
    # Cleanup: restore original laser voltage
    print(f"Restoring original laser PC voltage: {orig_laser_voltage:.3f} V")
    dl_pro_laser.set_pc_voltage(orig_laser_voltage)

def stop_automation():
    global _stop_flag
    _stop_flag = True
    print("Stopping flag set. Wait for current scan to abort.")


In [12]:
# Execute the sequence
run_automation()

In [13]:
# Emergency stop (run this in a new cell while the above is running)
stop_automation()